# Project Overview

This project focuses on predicting the star rating of company reviews using Natural Language Processing (NLP) and Deep Learning techniques. The dataset used for this project was obtained from the Kaggle competition "Sentiment Analysis - Company Reviews: Predict the number of stars given in company reviews". The objective is to build a model that can analyze the textual content of a review and accurately predict its rating on a scale of 1 to 5 stars.

To accomplish this, the review text was preprocessed, tokenized, and converted into numerical sequences. An LSTM-based neural network was then trained to learn patterns in the reviews and predict the corresponding star ratings.

Note: Original 'train dataset' used in 'explanation notebook' is used to split into train-validation-test for checking accuracy of the model

In [1]:
import pandas as pd
import numpy as np

In [2]:
df = pd.read_csv("C:\\Users\\bhavy\\OneDrive\\Desktop\\Projects\\Review_rating\\review_dataset.csv")

In [3]:
df.drop(['Id'],axis=1,inplace=True)

In [4]:
df.sample(2)

,Review,Rating
57815,The website is easy to use. I can find what I ...,5
24836,New washing machine delivered.\nWas contacted ...,5


In [5]:
df.shape

(60000, 2)

**We'll divide whole dataset into 64-16-20 split to also calculate model accuracy on test data**

In [6]:
from sklearn.model_selection import train_test_split

In [7]:
X_t,X_test,y_t,y_test = train_test_split(df['Review'],df['Rating'],stratify=df['Rating'],test_size=0.2,random_state=42)

In [8]:
X_train, X_validation,y_train,y_validation = train_test_split(X_t,y_t,stratify=y_t,test_size=0.2,random_state=42)

In [9]:
X_train.shape, X_validation.shape, X_test.shape

((38400,), (9600,), (12000,))

In [10]:
X_train.values[1]

'As described, arrived fast. Looks good'

In [11]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras.preprocessing.text import Tokenizer

In [12]:
tokenizer = Tokenizer(oov_token='New_word')

In [13]:
tokenizer.fit_on_texts(X_train)

In [14]:
tokenizer.word_index

{'New_word': 1,
 'the': 2,
 'to': 3,
 'and': 4,
 'i': 5,
 'a': 6,
 'was': 7,
 'my': 8,
 'for': 9,
 'it': 10,
 'of': 11,
 'they': 12,
 'with': 13,
 'in': 14,
 'on': 15,
 'have': 16,
 'is': 17,
 'not': 18,
 'service': 19,
 'that': 20,
 'me': 21,
 'this': 22,
 'as': 23,
 'you': 24,
 'had': 25,
 'be': 26,
 'very': 27,
 'no': 28,
 'but': 29,
 'from': 30,
 'delivery': 31,
 'so': 32,
 'phone': 33,
 'them': 34,
 'would': 35,
 'been': 36,
 'an': 37,
 'customer': 38,
 'time': 39,
 'at': 40,
 'all': 41,
 'will': 42,
 'good': 43,
 'are': 44,
 'when': 45,
 'up': 46,
 'order': 47,
 'get': 48,
 'great': 49,
 'now': 50,
 'we': 51,
 'out': 52,
 'again': 53,
 'after': 54,
 'company': 55,
 'day': 56,
 'which': 57,
 'their': 58,
 'if': 59,
 'one': 60,
 'told': 61,
 'do': 62,
 'back': 63,
 'then': 64,
 'just': 65,
 'days': 66,
 'what': 67,
 'were': 68,
 'use': 69,
 'new': 70,
 'still': 71,
 'by': 72,
 'easy': 73,
 'or': 74,
 'your': 75,
 'has': 76,
 'call': 77,
 'ordered': 78,
 'there': 79,
 'only': 80,
 '

In [15]:
total_vocab_len = len(tokenizer.word_index)

In [16]:
total_vocab_len

29911

In [17]:
int_encoded_X_train = tokenizer.texts_to_sequences(X_train)
int_encoded_X_val   = tokenizer.texts_to_sequences(X_validation)
int_encoded_X_test  = tokenizer.texts_to_sequences(X_test)

In [18]:
int_encoded_X_train[1]

[23, 624, 89, 136, 870, 43]

In [19]:
review_lengths=[]

for review in int_encoded_X_train:
    review_length = len(review)
    review_lengths.append(review_length)

In [20]:
print("Max length is: ",max(review_lengths))
print("90th percentile: ",np.percentile(review_lengths,90))
print("99.5th percentile: ",np.percentile(review_lengths,99.5))

Max length is:  1437
90th percentile:  133.0
99.5th percentile:  494.0049999999974


In [21]:
max_length = int(np.percentile(review_lengths,99.5))

In [22]:
from tensorflow.keras.utils import pad_sequences

In [23]:
X_train_padded = pad_sequences(int_encoded_X_train, maxlen=max_length,padding='post',truncating='post')
X_validation_padded = pad_sequences(int_encoded_X_val, maxlen=max_length,padding='post',truncating='post')
X_test_padded = pad_sequences(int_encoded_X_test, maxlen=max_length,padding='post',truncating='post')

In [24]:
X_train_padded[1]

array([ 23, 624,  89, 136, 870,  43,   0,   0,   0,   0,   0,   0,   0,
         0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,
         0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,
         0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,
         0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,
         0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,
         0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,
         0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,
         0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,
         0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,
         0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,
         0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,
         0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,
         0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   

In [25]:
y_train = y_train - 1

y_train.value_counts().sort_index(ascending=True)

Rating
0    11945
1     1042
2     1074
3     2144
4    22195
Name: count, dtype: int64

In [26]:
y_validation = y_validation - 1

# Creating the Neural network architecture

In [27]:
# tensorflow, keras, sequential, dense

from tensorflow import keras
from tensorflow.keras import Sequential
from tensorflow.keras.layers import Dense, LSTM, Embedding, Dropout

In [28]:
model = Sequential()

model.add(Embedding(input_dim=total_vocab_len+1,output_dim=128,input_shape=(max_length,)))

model.add(LSTM(units=64,dropout=0.2))

model.add(Dense(5,activation='softmax'))

C:\Users\bhavy\anaconda3\envs\pybase-TF2.0\lib\site-packages\keras\src\layers\core\embedding.py:93: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


In [29]:
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ (None, 494, 128)       │     3,828,736 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ (None, 128)            │       131,584 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 5)              │           645 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 3,960,965 (15.11 MB)

 Trainable params: 3,960,965 (15.11 MB)

 Non-trainable params: 0 (0.00 B)

# Compiling the network

In [30]:
model.compile(loss='sparse_categorical_crossentropy',optimizer='adam',metrics=['accuracy'])

# Training the network

In [31]:
from tensorflow.keras.callbacks import EarlyStopping

In [32]:
earlystop = EarlyStopping(monitor='val_loss',patience=6,restore_best_weights=True)

In [49]:
# model.fit(X_train_padded,y_train,epochs=20, batch_size=256,validation_data=(X_validation_padded, y_validation),callbacks=[earlystop])

**Model was fit using the above fit method. (Marked as markdown to prevent retraining the model when kernels are re-run)**

# Saving the model

In [50]:
# model.save("classification_review_rating_model.keras")

**saved after training**

In [51]:
import pickle

# Save tokenizer
with open("tokenizer.pkl", "wb") as f:
    pickle.dump(tokenizer, f)

# Save max_length
with open("max_length.pkl", "wb") as f:
    pickle.dump(max_length, f)

# Prediction

from tensorflow.keras.models import load_model

model = load_model("classification_review_rating_model.keras")

Loaded when kernels are run in next session

In [42]:
prob_predictions = model.predict(X_test_padded)

375/375 ━━━━━━━━━━━━━━━━━━━━ 20s 53ms/step


In [43]:
y_predictions = np.argmax(prob_predictions,axis=1)

In [44]:
y_predictions = y_predictions + 1

In [45]:
y_predictions

array([5, 5, 4, ..., 5, 5, 1])

# Checking metrics

In [46]:
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report, mean_absolute_error

In [47]:
print("Accuracy: ", accuracy_score(y_test,y_predictions))
print("\n")
print("MAE: ", mean_absolute_error(y_test,y_predictions))
print("\n")
print("Confusion matrix: ", confusion_matrix(y_test,y_predictions))
print("\n")
print("Classification report: ", classification_report(y_test,y_predictions))

Accuracy:  0.8596666666666667


MAE:  0.21825


Confusion matrix:  [[3527   26   78   66   35]
 [ 247    7   25   33   14]
 [ 138    6   46   93   53]
 [  57    6   43  129  435]
 [  62    6   50  211 6607]]


Classification report:                precision    recall  f1-score   support

           1       0.87      0.95      0.91      3732
           2       0.14      0.02      0.04       326
           3       0.19      0.14      0.16       336
           4       0.24      0.19      0.21       670
           5       0.92      0.95      0.94      6936

    accuracy                           0.86     12000
   macro avg       0.47      0.45      0.45     12000
weighted avg       0.83      0.86      0.84     12000



The model achieved high accuracy on ratings 1 and 5 but struggled with ratings 2, 3, and 4. This is mainly due to class imbalance in the dataset and the presence of mixed or overlapping sentiments in moderately rated reviews, making them harder to distinguish.

# Loading the model

In [48]:
#from tensorflow.keras.models import load_model
# model = load_model("classification_review_rating_model.keras")